# The neural estimator

A small convolutional network that predicts P(mine) for **one cell** from a 9×9
window around it. It approximates a solver that is already exact, so the point is
not accuracy — it is that inference costs the same however tangled the board is.

This notebook loads the trained weights that ship in the repository, runs them, and
measures them against the exact solver. Every number below is computed here.

The prose version is [`neural-estimator.md`](neural-estimator.md).

**Requires numpy and nothing else**, and the committed `neural/onnx/model.bin`.

In [1]:
import random, statistics, sys, time
sys.path.insert(0, '.')
sys.path.insert(0, '../neural')
import numpy as np
import minesweeper_ref as ms

net = ms.Network('../neural/onnx/model.bin')
print(f'{net.parameters:,} parameters (BatchNorm already folded in)')

121,665 parameters (BatchNorm already folded in)


## The shape of it

Four 3×3 convolutions with BatchNorm and ReLU, an average pool down to 3×3, the
global mines ratio appended back on, then 289 → 128 → 64 → 1 and a sigmoid.

Two details that are not decoration:

**The ratio is concatenated after the pooling.** It arrives as channel 6, constant
across the patch, so pooling preserves it — but only as one value diluted among 288.
Appending it gives the head an undiluted copy. It is the only non-local thing in the
input, and on a board where no number speaks it is the *only* signal there is.

**The receptive field is the whole patch.** Four padded 3×3 convolutions reach nine
cells across. A mine constrains only its eight neighbours, and those neighbours'
numbers all lie within four cells of the target — so the network can in principle see
every constraint that bears on the cell, and nothing at all about the rest of the
board except channel 6.

In [2]:
reach = 1
for i, (w, b) in enumerate(net.convs):
    reach += 2
    print(f'conv{i}  weight {str(w.shape):18} bias {str(b.shape):8} '
          f'{w.size + b.size:>7,} params   receptive field {reach}x{reach}')
print()
for i, (w, b) in enumerate(net.linears):
    print(f'fc{i}    weight {str(w.shape):18} bias {str(b.shape):8} {w.size + b.size:>7,} params')
print()
print(f'patch is 9x9, so the last convolution sees all of it')

conv0  weight (32, 8, 3, 3)      bias (32,)      2,336 params   receptive field 3x3
conv1  weight (64, 32, 3, 3)     bias (64,)     18,496 params   receptive field 5x5
conv2  weight (64, 64, 3, 3)     bias (64,)     36,928 params   receptive field 7x7
conv3  weight (32, 64, 3, 3)     bias (32,)     18,464 params   receptive field 9x9

fc0    weight (128, 289)         bias (128,)    37,120 params
fc1    weight (64, 128)          bias (64,)      8,256 params
fc2    weight (1, 64)            bias (1,)          65 params

patch is 9x9, so the last convolution sees all of it


## What the network is shown

Eight channels over a 9×9 window centred on the cell in question.

| Ch | Value | Meaning |
|----|-------|---------|
| 0 | number ÷ 8 | what a visible cell shows |
| 1 | {0,1} | is visible |
| 2 | {0,1} | is hidden |
| 3 | {0,1} | is flagged |
| 4 | {0,1} | off the edge of the board |
| 5 | 1 at (4,4) | *this* is the cell being asked about |
| 6 | ratio | mines remaining ÷ cells still hidden, broadcast |
| 7 | {0,1} | unopened and touching a visible number |

Two traps live in there, and neither raises an error — get one wrong and the model is
simply fed an input it never saw:

- **Channel 6's denominator counts only `Hidden` cells, not flagged ones**, and its
  numerator subtracts mines actually *uncovered*, not flags the player has placed.
  (The solver treats a flag as an unknown; the ratio treats it as spoken for. They
  are answering different questions.)
- **Channel 7 needs a visible neighbour showing a number greater than zero.** A
  visible `0` constrains nothing, so it does not make its neighbours border cells.

Both of those are mistakes I made writing `minesweeper_ref.py`, and both were caught
only by the cross-check two cells further down.

In [3]:
board = ms.from_text("""
...o**ooo
...*..oo.
......*oo
.......oo
*...*..oo
....*...*
""")
print(board.render())

target = (5, 1)
p = ms.patch(board, *target)
names = ['number/8', 'visible', 'hidden', 'flagged', 'off-board', 'target', 'ratio', 'border']
print(f'\npatch for {target}, shape {p.shape}\n')
for c in (0, 1, 2, 5, 7):
    print(f'channel {c}  {names[c]}')
    for row in p[c]:
        print('  ' + ''.join(('.' if v == 0 else f'{v:g}'[:4]).rjust(5) for v in row))
    print()
print(f'channel 6  {names[6]} = {p[6][0][0]:.4f} everywhere')

    ▒    ▒    ▒    2    ▒    ▒    1    ·    ·
    ▒    ▒    ▒    ▒    ▒    ▒    2    1    ▒
    ▒    ▒    ▒    ▒    ▒    ▒    ▒    1    ·
    ▒    ▒    ▒    ▒    ▒    ▒    ▒    1    ·
    ▒    ▒    ▒    ▒    ▒    ▒    ▒    1    1
    ▒    ▒    ▒    ▒    ▒    ▒    ▒    ▒    ▒

patch for (5, 1), shape (8, 9, 9)

channel 0  number/8
      .    .    .    .    .    .    .    .    .
      .    .    .    .    .    .    .    .    .
      .    .    .    .    .    .    .    .    .
      .    . 0.25    .    . 0.12    .    .    .
      .    .    .    .    . 0.25 0.12    .    .
      .    .    .    .    .    . 0.12    .    .
      .    .    .    .    .    . 0.12    .    .
      .    .    .    .    .    . 0.12 0.12    .
      .    .    .    .    .    .    .    .    .

channel 1  visible
      .    .    .    .    .    .    .    .    .
      .    .    .    .    .    .    .    .    .
      .    .    .    .    .    .    .    .    .
      .    .    1    .    .    1    1    1    .
      .    .    .    .  

## Checking the patch against the engine

The patch layout is written down in three places — `probability/patch.rs`,
`neural/dataset.py`, and `minesweeper_ref.py` — with nothing tying them together.

`rust_reference.txt` is the tie. It holds the probabilities the **Rust engine**
produced for one fixed position; if the Python built a different patch, the numbers
would not match. (They agree to float32 rounding, ~5e-7.)

In [4]:
fixture = ms.from_text("""
*.....
..*...
....*.
...*..
""")
# ...with the mine at (5,0) and the opened/flagged squares the fixture describes.
fixture.mines = {(0, 0), (2, 1), (3, 3), (5, 0), (4, 2)}
for cell in [(1, 1), (2, 2), (3, 1), (4, 1), (1, 2), (3, 2)]:
    fixture.state[cell] = ms.VISIBLE
fixture.state[(1, 0)] = ms.FLAGGED

rust = {}
for line in open('rust_reference.txt'):
    if line.startswith('#') or not line.strip():
        continue
    x, y, value = line.split()
    rust[(int(x), int(y))] = float(value)

worst = max(abs(net.predict(ms.patch(fixture, *c)) - rust[c]) for c in fixture.hidden_cells())
print(f'compared {len(fixture.hidden_cells())} cells against the Rust engine')
print(f'worst disagreement: {worst:.2e}')
assert worst < 1e-4, 'the Python patch does not match the one the engine builds'

compared 18 cells against the Rust engine
worst disagreement: 5.02e-07


And the network itself against `patchcnn_reference.py` — the deliberately
unoptimised numpy version that serves as the written specification. `Network` above
is the same arithmetic vectorised, so it has to agree.

In [5]:
from patchcnn_reference import forward

cells = board.hidden_cells()[:6]
worst = max(abs(net.predict(ms.patch(board, *c)) -
                forward('../neural/onnx/model.bin', ms.patch(board, *c))) for c in cells)
print(f'vectorised vs the reference implementation: {worst:.2e} over {len(cells)} cells')

t = time.time(); net.score_board(board); fast = time.time() - t
t = time.time()
for c in cells:
    forward('../neural/onnx/model.bin', ms.patch(board, *c))
slow = (time.time() - t) / len(cells) * len(board.hidden_cells())
print(f'whole board: {fast:.3f}s vectorised, ~{slow:.1f}s with the reference '
      f'(which re-reads the weights every call)')

vectorised vs the reference implementation: 2.68e-07 over 6 cells


whole board: 0.084s vectorised, ~7.0s with the reference (which re-reads the weights every call)


## Side by side with the exact solver

The same board, scored both ways. This is what the web front-end shows: the proof in
one corner of each cell, the guess in the other.

In [6]:
exact = ms.exact_probabilities(board)
guess = net.score_board(board)

print('exact (constraint search)')
print(board.render(exact))
print()
print('network')
print(board.render(guess))
print()
gap = {c: abs(exact[c] - guess[c]) for c in exact}
print(f'mean |network - exact| = {statistics.mean(gap.values()):.3f}, '
      f'worst {max(gap.values()):.3f} at {max(gap, key=gap.get)}')
print(f'the network\'s numbers sum to {sum(guess.values()):.1f}; there are '
      f'{board.mines_count} mines — nothing constrains it to be calibrated that way')

exact (constraint search)
  11%  11%  40%    2  40%  50%    1    ·    ·
  11%  11%  40%  40%  40%  50%    2    1   0%
  11%  11%  11%  11%  11%   0% 100%    1    ·
  11%  11%  11%  11%  11%  11%   0%    1    ·
  11%  11%  11%  11%  11%  11%   0%    1    1
  11%  11%  11%  11%  11%  11%   0%  50%  50%

network
  17%  19%  41%    2  32%  78%    1    ·    ·
  18%  18%  38%  39%  29%  94%    2    1   2%
  17%  16%  15%  14%  17%   1% 100%    1    ·
  17%  17%  15%  16%  16%  15%   0%    1    ·
  18%  18%  17%  15%  17%  14%   0%    1    1
  18%  18%  16%  15%  17%  15%   0%  35%  32%

mean |network - exact| = 0.066, worst 0.443 at (5, 1)
the network's numbers sum to 9.7; there are 8 mines — nothing constrains it to be calibrated that way


The last line is worth pausing on. The solver's numbers *must* sum to the mine
count — it is counting layouts of exactly that many mines. The network has no such
constraint: it scores each cell independently and nothing ties the total to
anything. That it lands close is a property it learned, not one it is given.

## How good is it, measured here

The number that matters is not the mean error. It is what the network says about
cells the solver has **proved**, because those are the ones auto-play would act on:

- a proven-safe cell the network scores high is a wasted move;
- a proven-mine cell the network scores low is a **lost game**.

Both sweeps below generate real positions, solve them exactly, and compare.
(A minute or so; the exact solver here enumerates outright, so positions with a very
large group are skipped rather than waited for.)

In [7]:
def sweep(width, height, mines, seed, want, budget, max_group=16):
    random.seed(seed)
    safe, mine, positions = [], [], 0
    started = time.time()
    while positions < want and time.time() - started < budget:
        squares = [(x, y) for y in range(height) for x in range(width)]
        laid = set(random.sample(squares, mines))
        b = ms.Board(width, height, laid)
        b.reveal(*random.choice([c for c in squares if c not in laid]))

        hidden, constraints, to_place = ms.build_constraints(b)
        proven_mines, proven_safe, reduced, _ = ms.propagate(len(hidden), constraints, to_place)
        decided = set(proven_mines) | set(proven_safe)
        free = [i for i in range(len(hidden)) if i not in decided]
        remap = {c: i for i, c in enumerate(free)}
        groups = ms.decompose([([remap[c] for c in cs], r) for cs, r in reduced], len(free))
        if groups and max(len(cells) for cells, _ in groups) > max_group:
            continue          # would take longer to enumerate than it is worth

        exact = ms.exact_probabilities(b)
        if not exact:
            continue
        scored = net.score_board(b)
        counted = False
        for cell, p in exact.items():
            if p == 0.0:
                safe.append(scored[cell]); counted = True
            elif p == 1.0:
                mine.append(scored[cell]); counted = True
        positions += counted
    return positions, safe, mine, time.time() - started

share = lambda values, test: 100 * sum(1 for v in values if test(v)) / max(len(values), 1)

for width, height, mines, label in [(9, 9, 10, 'beginner'), (16, 16, 40, 'intermediate')]:
    n, safe, mine, elapsed = sweep(width, height, mines, 7, want=40, budget=45)
    print(f'{label}  {width}x{height}/{mines}  —  {n} positions, {elapsed:.0f}s')
    print(f'   solver says SAFE ({len(safe):4d} cells): network median {statistics.median(safe):.3f}, '
          f'{share(safe, lambda v: v < 0.05):4.1f}% under 5%, worst call {max(safe):.3f}')
    print(f'   solver says MINE ({len(mine):4d} cells): network median {statistics.median(mine):.3f}, '
          f'{share(mine, lambda v: v > 0.95):4.1f}% over 95%, worst call {min(mine):.3f}')
    print()

beginner  9x9/10  —  40 positions, 14s
   solver says SAFE ( 275 cells): network median 0.000, 89.8% under 5%, worst call 0.558
   solver says MINE ( 213 cells): network median 1.000, 94.8% over 95%, worst call 0.403



intermediate  16x16/40  —  18 positions, 45s
   solver says SAFE ( 207 cells): network median 0.000, 84.1% under 5%, worst call 0.588
   solver says MINE ( 134 cells): network median 1.000, 90.3% over 95%, worst call 0.184



Two things to read off that.

**It is well calibrated in the middle and wrong at the edges.** The medians are
essentially 0 and 1 — on a typical proved cell the network agrees completely. The
*worst* calls are the story: a proven-safe cell scored above 0.5, a proven-mine cell
scored under 0.25. Those are rare and they are catastrophic, which is why the exact
solver stays authoritative and the guess never feeds auto-reveal.

**It degrades with density.** The held-out test split gives 0.143% of proven mines
called under 10%. Mid-game Expert positions measured while building the web
front-end were far worse: only 76.5% of proven-safe cells under 5%, against ~90%
here. It is a *local* approximation, and a dense board needs more than local
information.

## Correction during play

Every exact solve is a perfectly labelled position that cost nothing extra to
produce. So the web front-end steps the network's **output layer** towards the
solver's answers after each solve.

Only the last layer. With a sigmoid output and cross-entropy loss the gradient into
the head's weights is just `(prediction − target) × feature` — no chain rule left to
apply — which is a dozen lines of Rust rather than an autograd engine in the browser.

And the forward pass it needs is the same one `predict` does, which is why the page
folds correction into the scoring pass instead of running a second one. Done
separately over a whole board it measured **4.2 s on 40×40, on every move**.

In [8]:
fresh = ms.Network('../neural/onnx/model.bin')
targets = ms.exact_probabilities(board)
cells = list(targets)

def mean_gap(network):
    return statistics.mean(abs(network.predict(ms.patch(board, *c)) - targets[c]) for c in cells)

print(f'before      : mean |network - exact| = {mean_gap(fresh):.4f}')
for step in range(1, 4):
    for c in cells:
        fresh.learn(ms.patch(board, *c), targets[c], rate=0.02)   # the rate the page uses
    print(f'after {step} pass{"es" if step > 1 else "  "}: mean |network - exact| = {mean_gap(fresh):.4f}')

print()
print('this position, corrected')
print(board.render({c: fresh.predict(ms.patch(board, *c)) for c in cells}))

before      : mean |network - exact| = 0.0657


after 1 pass  : mean |network - exact| = 0.0522


after 2 passes: mean |network - exact| = 0.0503


after 3 passes: mean |network - exact| = 0.0501

this position, corrected


  14%  16%  36%    2  29%  69%    1    ·    ·
  15%  15%  34%  34%  26%  90%    2    1   2%
  14%  13%  12%  11%  14%   1% 100%    1    ·
  14%  14%  12%  13%  13%  12%   0%    1    ·
  15%  15%  14%  12%  13%  11%   0%    1    1
  15%  15%  13%  12%  14%  12%   0%  28%  26%


### The rate is not a free parameter

Because only the head moves, the update is `w -= rate * error * features`, and the
shift it produces in the logit is `rate * error * |features|²`. Those features are
post-ReLU activations of a 64-wide layer, so `|features|²` is around 120 — which
means a rate that looks small is not.

At 0.5 the correction destroys the model in one pass. The page uses **0.02**.

In [9]:
print(f'|features|^2 for one cell: {float(fresh.features(ms.patch(board, *cells[0])) @ fresh.features(ms.patch(board, *cells[0]))):.0f}')
print()
print(f"{'rate':>8}  {'before':>8}  {'1 pass':>8}  {'2':>8}  {'3':>8}")
for rate in (0.5, 0.1, 0.02, 0.005, 0.001):
    net_r = ms.Network('../neural/onnx/model.bin')
    before = statistics.mean(abs(net_r.predict(ms.patch(board, *c)) - targets[c]) for c in cells)
    after = []
    for _ in range(3):
        for c in cells:
            net_r.learn(ms.patch(board, *c), targets[c], rate)
        after.append(statistics.mean(abs(net_r.predict(ms.patch(board, *c)) - targets[c]) for c in cells))
    marks = '  '.join(f'{v:8.4f}' for v in after)
    note = '   <- the page' if rate == 0.02 else ('   <- destroys it' if rate == 0.5 else '')
    print(f'{rate:>8}  {before:8.4f}  {marks}{note}')

|features|^2 for one cell: 123

    rate    before    1 pass         2         3


     0.5    0.0657    0.5010    0.6368    0.6081   <- destroys it


     0.1    0.0657    0.0937    0.0958    0.0935


    0.02    0.0657    0.0522    0.0503    0.0501   <- the page


   0.005    0.0657    0.0580    0.0536    0.0509


   0.001    0.0657    0.0637    0.0618    0.0603


### Does it damage the boards it is *not* looking at?

A sequence of single-position updates is the classic recipe for catastrophic
forgetting, so this is worth checking rather than assuming. Correct on four boards
in turn and watch the error on four others the network never sees.

In [10]:
random.seed(3)

def sample_position(width, height, mines, max_group=14):
    while True:
        squares = [(x, y) for y in range(height) for x in range(width)]
        laid = set(random.sample(squares, mines))
        b = ms.Board(width, height, laid)
        b.reveal(*random.choice([c for c in squares if c not in laid]))
        hidden, constraints, to_place = ms.build_constraints(b)
        pm, ps, reduced, _ = ms.propagate(len(hidden), constraints, to_place)
        decided = set(pm) | set(ps)
        free = [i for i in range(len(hidden)) if i not in decided]
        remap = {c: i for i, c in enumerate(free)}
        groups = ms.decompose([([remap[c] for c in cs], r) for cs, r in reduced], len(free))
        if groups and max(len(cs) for cs, _ in groups) > max_group:
            continue
        exact = ms.exact_probabilities(b)
        if exact:
            return b, exact

positions = [sample_position(9, 9, 10) for _ in range(8)]
train, held_out = positions[:4], positions[4:]

def gap(network, b, exact):
    return statistics.mean(abs(network.predict(ms.patch(b, *c)) - exact[c]) for c in exact)

net_c = ms.Network('../neural/onnx/model.bin')
print(f'held-out error before any correction : '
      f'{statistics.mean(gap(net_c, b, e) for b, e in held_out):.4f}')
for i, (b, exact) in enumerate(train):
    for c in exact:
        net_c.learn(ms.patch(b, *c), exact[c], 0.02)
    print(f'  after correcting on board {i}: this board {gap(net_c, b, exact):.4f}   '
          f'held-out {statistics.mean(gap(net_c, hb, he) for hb, he in held_out):.4f}')

held-out error before any correction : 0.0160


  after correcting on board 0: this board 0.0045   held-out 0.0144


  after correcting on board 1: this board 0.0067   held-out 0.0097


  after correcting on board 2: this board 0.0099   held-out 0.0108


  after correcting on board 3: this board 0.0053   held-out 0.0122


It generalises rather than overfitting: the boards it never saw improve too. That is
the case for doing it at all — and the reason to keep the rate where it is.

Note the page does one pass per position at 0.02, which is a nudge, not training.

Two gates on that, in the page, both for a reason:

- **Only after an exact solve.** A sampled estimate carries noise, and a network
  taught from noise learns the noise.
- **Only in the display modes where the solver is visible.** Under *neural network
  only* the page runs uncorrected, because a network being corrected by the solver
  mid-run is not the thing that mode is there to measure.

Note also that the step above overfits this one position — press it far enough and
the network gets very good at this board and worse everywhere else. In the page the
rate is 0.02 and each position is seen once, which is a nudge, not training.

## Training

`datagen` (Rust) plays real games and labels them with the exact constraint solver.
Configurations are sampled uniformly from 9×9/10, 10×10/15, 16×16/40, 16×16/51 and
30×16/99, and each game is played a random number of moves before being labelled, so
positions come from every stage rather than only the end.

Two deliberate choices:

- **A position the solver cannot finish within budget is skipped, not sampled.** In
  the output a sampled label is indistinguishable from an exact one, so letting one
  through would quietly teach the model somebody's guesses.
- **Labels are probabilities, not mine/no-mine.** A cell the solver puts at 0.4 should
  be predicted 0.4, not rounded to "safe".

**60 000 positions → 6 485 906 training cells**, 757 394 validation, ~380 000 test.

| | |
|---|---|
| Loss | `BCE + 0.1 · MSE` |
| Optimiser | AdamW, lr 1e-3, weight decay 1e-4, cosine annealing |
| Batch | 2048 |
| Augmentation | D4 — 4 rotations × 2 flips; labels are symmetry-invariant |
| Epochs | 10, on CPU |

BCE gives calibration: it punishes a confident wrong answer far harder than a hedged
one, which for a probability is exactly right. The MSE term is small and only
discourages being wrong by a lot.

```
Epoch   1/10  train_loss=0.4868  val_bce=0.4906  val_mae=0.0800
Epoch   2/10  train_loss=0.4586  val_bce=0.4474  val_mae=0.0477
Epoch   3/10  train_loss=0.4414  val_bce=0.4324  val_mae=0.0356
Epoch   4/10  train_loss=0.4328  val_bce=0.4274  val_mae=0.0293
Epoch   5/10  train_loss=0.4300  val_bce=0.4260  val_mae=0.0296
Epoch   6/10  train_loss=0.4280  val_bce=0.4257  val_mae=0.0348
Epoch   7/10  train_loss=0.4269  val_bce=0.4241  val_mae=0.0300
Epoch   8/10  train_loss=0.4261  val_bce=0.4243  val_mae=0.0334
Epoch   9/10  train_loss=0.4256  val_bce=0.4229  val_mae=0.0274   <- best
Epoch  10/10  train_loss=0.4253  val_bce=0.4231  val_mae=0.0298
```

**This is a first run, not a tuned one.** The curve was still drifting down when
cosine annealing took the learning rate to zero. Ten epochs was what fit; it is the
most obvious thing to improve, and the 0.143% is the number to watch while doing it.

`neural/README.md` records the four separate reasons this would not train before —
an 11.5 GB dataset representation, training data that was all endgames, a Rust
feature that did not compile, and a missing `onnxscript`. None of them the model.

## Two implementations, on purpose

`model.onnx` is read by `tract` in the desktop build. `model.bin` is the same weights
with BatchNorm folded into the preceding convolution, read by
`probability::patch_cnn` — the forward pass written out by hand in Rust.

The hand-written one exists for two reasons. **Size**: `tract` costs 13.4 MB in the
WebAssembly build against a 174 KB baseline. **And the gradient**: owning the
arithmetic means owning the derivative, which is what makes the correction above
possible in the browser.

At inference BatchNorm is an affine map with fixed parameters, so it folds away:

```
scale = gamma / sqrt(var + eps)
W'    = W * scale
b'    = (b - mean) * scale + beta
```

That is the difference between 122 049 trainable parameters and the 121 665 in the
file — the 384 BatchNorm scale and shift values are gone, absorbed.

Three implementations therefore have to agree, so `export_weights.py` writes
`model.vectors`: patches with the outputs PyTorch gave for them, letting the Rust
test verify itself with no Python present. The notebook can read it too.

In [11]:
raw = open('../neural/onnx/model.vectors', 'rb').read()
assert raw[:8] == b'MSPCNNV1'
count = int(np.frombuffer(raw[8:12], dtype='<u4')[0])
values = np.frombuffer(raw, dtype='<f4', offset=12).reshape(count, 8 * 9 * 9 + 1)

worst = max(abs(net.predict(row[:-1].reshape(8, 9, 9)) - row[-1]) for row in values)
print(f'{count} patches with the outputs PyTorch gave for them')
print(f'worst disagreement with PyTorch: {worst:.2e}')
assert worst < 1e-4

24 patches with the outputs PyTorch gave for them
worst disagreement with PyTorch: 1.97e-06


So four readings of this network now agree, to float32 rounding: PyTorch (through
the recorded vectors), `patchcnn_reference.py`, the Rust that ships, and the code in
this notebook.

## Where this lives

| What | Where |
|---|---|
| The model | `neural/model.py` |
| Patch layout | `minesweeper_core/src/probability/patch.rs`, `neural/dataset.py` |
| Data generation | `datagen/src/main.rs`, `neural/datagen.py` |
| Training | `neural/train.py`, `neural/prepare.py` |
| Export | `neural/export.py`, `neural/export_weights.py` |
| The numpy specification | `neural/patchcnn_reference.py` |
| Rust inference and the gradient step | `minesweeper_core/src/probability/patch_cnn.rs` |
| Tests | `minesweeper_core/tests/patch_cnn.rs`, `tests/neural.rs` |

The solver it is approximating is described in
[`constraint-estimator.md`](constraint-estimator.md) and
[`constraint-estimator.ipynb`](constraint-estimator.ipynb).